In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", "true").load("/Volumes/external-catalog/default/test-ext-vol/Employee_Attrition.csv")
display(df)

In [0]:
# Filter high risk attrition employees
high_risk_df = df.filter(
    (df["Attrition"] == "No") & (df["JobSatisfaction"].cast("int") < 3)
)

# Select relevant informative columns
selected_df = high_risk_df.select(
    "EmployeeNumber", "Attrition", "JobSatisfaction", "Department", "Age", "MonthlyIncome"
)

# Write the transformed data into a Delta table within the default schema of external-catalog
selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
display(spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_attrition_employees"))

In [0]:
spark.sql("""
INSERT INTO `external-catalog`.default.high_risk_attrition_employees (EmployeeNumber, Attrition, JobSatisfaction, Department, Age, MonthlyIncome)
VALUES (99999, 'No', 1, 'DummyDept', 30, 1000)
""")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history()
display(history_df.select("version","operation"))

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
df_version = spark.read.option("versionAsOf", 0).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_version)

In [0]:
df_timestamp = spark.read.option("timestampAsOf", "2025-12-14 13:31:24.724+00:00").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_timestamp)

In [0]:
spark.sql("""
CREATE VOLUME `external-catalog`.default.employee_transformed_data
""")

In [0]:
# Logical transformation: filter employees with MonthlyIncome > 3000 and Age < 40
transformed_df = df.filter(
    (df["MonthlyIncome"].cast("int") > 3000) & (df["Age"].cast("int") < 40)
)

# Write to volume partitioned by Department
transformed_df.write.partitionBy("Department").format("parquet").mode("overwrite").save("/Volumes/external-catalog/default/employee_transformed_data")